# Accessibility Inequality Analysis — Theil Decomposition

This notebook computes the population-weighted **Theil index**, which splits total
inequality into a *between-group* and a *within-group* component. The
decomposition is run along three dimensions: **municipality**, **socio-economic
status** (SES tercile), and **national origin** (national / other-EU / non-EU).



## Folder structure

All input files live in a `data/` folder next to this notebook. Run the
notebook from its own directory so the relative paths below resolve correctly.

```text
theil_index/
├── compute_theil.ipynb                        # this notebook — Theil decomposition
├── create_grid_to_municipality.ipynb          # run this first to build grid_to_municipality.csv
└── data/
    ├── accessibility_results_sigma_20p0.csv   # gravity based accessibility scores 
    ├── origins.csv                            # population grid
    ├── lux_census_2021_socio.csv              # 2021 census nationality breakdown
    └── grid_to_municipality.csv               # created by create_grid_to_municipality.ipynb
```


## Required columns per input file

| File | Required columns | Notes |
|------|------------------|-------|
| `accessibility_results_sigma_20p0.csv` | `GRD_ID`, and `A_<category>_A` + `A_<category>_D` for every category | |
| `origins.csv` | `GRD_ID`, `Pop_grids` | `Pop_grids` is the cell population; privacy-suppressed cells may hold `<30` (parsed to a midpoint of 15). |
| `lux_census_2021_socio.csv` | `SPATIAL`, `NAT`, `EU_OTH`, `OTH` | `SPATIAL` is joined to `GRD_ID`. `NAT` / `EU_OTH` / `OTH` are national / other-EU / non-EU population counts; NA or all-zero rows are imputed. |
| `grid_to_municipality.csv` | `GRD_ID`, `municipality`, `ses_tercile` | |


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re


In [2]:
# ── CONFIG ──────────────────────────────────────────────────────────────

PROJECT_ROOT = Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"

ACC_PATH     = str(DATA_DIR / "accessibility_results_sigma_20p0.csv")
ORIGINS_PATH = str(DATA_DIR / "origins.csv")                 # population grid
CENSUS_PATH  = str(DATA_DIR / "lux_census_2021_socio.csv")   # nationality breakdown
MUNI_PATH    = str(DATA_DIR / "grid_to_municipality.csv")    # GRD_ID, municipality, ses_tercile
SCENARIOS  = ["A", "D"]
CATEGORIES = [
    "entertainment", "jobs", "commercial", "bars_and_restaurants",
    "food", "education", "parks", "basic_healthcare"
]
EPS = 1e-9
SUPPRESSED_MIDPOINT = 15  # midpoint for <30 cells



In [3]:
# ── LOAD ACCESSIBILITY ──────────────────────────────────────────────────
acc = pd.read_csv(ACC_PATH)

print("done")
# ── MEAN-NORMALIZATION ACROSS SCENARIOS ─────────────────────────────────

for cat in CATEGORIES:
    raw_A = f"A_{cat}_A"
    denom = acc[raw_A].mean()
    if denom <= 0 or not np.isfinite(denom):
        for scen in SCENARIOS:
            acc[f"A_{cat}_{scen}_norm"] = 1.0
        continue
    for scen in SCENARIOS:
        raw_col = f"A_{cat}_{scen}"
        acc[f"{raw_col}_norm"] = acc[raw_col] / denom

# ── AGGREGATE NORMALIZED ACCESSIBILITY ─────────────────────────────────
for scen in SCENARIOS:
    norm_cols = [f"A_{cat}_{scen}_norm" for cat in CATEGORIES]
    acc[f"A_agg_{scen}"] = acc[norm_cols].mean(axis=1)

# ── LOAD ORIGINS  ──────────────────────
# Parse Pop_grids: exact numbers kept as-is; '<30' cells → midpoint of 15.
def parse_pop_value(x, midpoint=SUPPRESSED_MIDPOINT):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    try:
        return float(s)
    except ValueError:
        pass
    if re.match(r"^\s*<=?\s*30\s*$", s):
        return float(midpoint)
    m = re.search(r"(\d+(\.\d+)?)", s)
    if m:
        return float(m.group(1))
    return np.nan

origins = pd.read_csv(ORIGINS_PATH)
origins["T"] = origins["Pop_grids"].apply(parse_pop_value)
origins["is_suppressed"] = origins["Pop_grids"].apply(
    lambda x: bool(re.match(r"^\s*<=?\s*30\s*$", str(x).strip())) if isinstance(x, str) else False
)

n_suppressed = origins["is_suppressed"].sum()
print(f"Total cells in origins:       {len(origins)}")
print(f"Suppressed cells (<30):       {n_suppressed} ({n_suppressed/len(origins)*100:.1f}%)")
print(f"Total population (origins):   {origins['T'].sum():,.0f}")



done
Total cells in origins:       2795
Suppressed cells (<30):       567 (20.3%)
Total population (origins):   646,170


In [4]:
# ── LOAD CENSUS (nationality breakdown only) ─────────────────────────────
census = pd.read_csv(CENSUS_PATH)

# ── COMPUTE NATIONAL AVERAGE SHARES from non-suppressed, complete cells ──
# These shares are used to impute NAT/EU_OTH/OTH for suppressed cells
# where the census reports NAs or zero population.
census_complete = census[
    census["NAT"].notna() &
    census["EU_OTH"].notna() &
    census["OTH"].notna()
].copy()
census_complete["T_census"] = census_complete[["NAT", "EU_OTH", "OTH"]].sum(axis=1)
census_complete = census_complete[census_complete["T_census"] > 0]

total_complete = census_complete["T_census"].sum()
nat_share   = census_complete["NAT"].sum()    / total_complete
eu_oth_share= census_complete["EU_OTH"].sum() / total_complete
oth_share   = census_complete["OTH"].sum()    / total_complete

print(f"\nNational average shares (from {len(census_complete)} complete cells):")
print(f"  NAT:    {nat_share:.3f}")
print(f"  EU_OTH: {eu_oth_share:.3f}")
print(f"  OTH:    {oth_share:.3f}")

# ── MERGE ORIGINS → CENSUS ───────────────────────────────────────────────
pop = origins[["GRD_ID", "T", "is_suppressed"]].merge(
    census[["SPATIAL", "NAT", "EU_OTH", "OTH"]],
    left_on="GRD_ID", right_on="SPATIAL", how="left"
)

# ── IMPUTE NAT/EU_OTH/OTH FOR PROBLEMATIC CELLS ─────────────────────────
# A cell needs imputation when:
#   (a) census NAT/EU_OTH/OTH are NA, OR
#   (b) census NAT+EU_OTH+OTH = 0  (census treats cell as uninhabited)

pop["T_census"] = pop[["NAT", "EU_OTH", "OTH"]].sum(axis=1)
needs_imputation = pop["NAT"].isna() | (pop["T_census"] == 0)

pop.loc[needs_imputation, "NAT"]    = pop.loc[needs_imputation, "T"] * nat_share
pop.loc[needs_imputation, "EU_OTH"] = pop.loc[needs_imputation, "T"] * eu_oth_share
pop.loc[needs_imputation, "OTH"]    = pop.loc[needs_imputation, "T"] * oth_share

n_imputed = needs_imputation.sum()
print(f"\nCells imputed (NA or zero in census): {n_imputed}")
print(f"Population in imputed cells:          {pop.loc[needs_imputation, 'T'].sum():,.0f}")
print(f"Total population after imputation: {pop['T'].sum():,.0f}")




National average shares (from 985 complete cells):
  NAT:    0.501
  EU_OTH: 0.341
  OTH:    0.158

Cells imputed (NA or zero in census): 1508
Population in imputed cells:          5,367
Total population after imputation: 646,170


In [5]:
# ── LOAD MUNICIPALITY ────────────────────────────────────────────────────
muni = pd.read_csv(MUNI_PATH)

# ── MERGE ALL ────────────────────────────────────────────────────────────
df = acc.merge(pop[["GRD_ID", "T", "NAT", "EU_OTH", "OTH"]], on="GRD_ID", how="inner")
df = df.merge(muni, on="GRD_ID", how="inner")

required_cols = ["GRD_ID", "T", "municipality", "ses_tercile", "NAT", "EU_OTH", "OTH"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns after merge: {missing}")

print(f"\nCells in final df: {len(df)}")
print(f"A_agg_A: mean={df['A_agg_A'].mean():.4f}, "
      f"min={df['A_agg_A'].min():.4f}, max={df['A_agg_A'].max():.4f}")
print(f"A_agg_D: mean={df['A_agg_D'].mean():.4f}, "
      f"min={df['A_agg_D'].min():.4f}, max={df['A_agg_D'].max():.4f}")



Cells in final df: 2795
A_agg_A: mean=1.0000, min=0.0000, max=21.6208
A_agg_D: mean=1.0170, min=0.0000, max=22.8354


In [6]:
# ── THEIL FUNCTIONS ─────────────────────────────────────────────────────
def weighted_theil_total(data, access_col, weight_col):
    d = data[[access_col, weight_col]].dropna().copy()
    d = d[d[weight_col] > 0].copy()
    if len(d) == 0:
        return None
    d[access_col] = d[access_col].clip(lower=EPS)
    W = d[weight_col].sum()
    mu = (d[access_col] * d[weight_col]).sum() / W
    ratio = d[access_col] / mu
    T = (d[weight_col] * ratio * np.log(ratio)).sum() / W
    return {"T_total": T, "n_cells": len(d), "n_pop": W}


def theil_decompose(data, access_col, group_col, weight_col):
    d = data[[access_col, group_col, weight_col]].dropna().copy()
    d = d[d[weight_col] > 0].copy()
    if len(d) == 0:
        return None
    d[access_col] = d[access_col].clip(lower=EPS)

    W = d[weight_col].sum()
    mu = (d[access_col] * d[weight_col]).sum() / W
    ratio = d[access_col] / mu
    T_total = (d[weight_col] * ratio * np.log(ratio)).sum() / W

    T_between = 0.0
    T_within_direct = 0.0
    for g, sub in d.groupby(group_col):
        Wg = sub[weight_col].sum()
        if Wg <= 0:
            continue
        mu_g = (sub[access_col] * sub[weight_col]).sum() / Wg
        mu_g = max(mu_g, EPS)
        share_g = Wg / W
        r_g = mu_g / mu
        T_between += share_g * r_g * np.log(r_g)
        ratio_g = sub[access_col] / mu_g
        T_g = (sub[weight_col] * ratio_g * np.log(ratio_g)).sum() / Wg
        T_within_direct += share_g * r_g * T_g

    T_within = T_total - T_between
    return {
        "T_total": T_total,
        "T_between": T_between,
        "T_within": T_within,
        "T_within_direct": T_within_direct,
        "between_share_pct": (T_between / T_total * 100) if T_total > 0 else 0.0,
        "within_share_pct": (T_within / T_total * 100) if T_total > 0 else 0.0,
        "n_cells": len(d),
        "n_pop": W,
        "n_groups": d[group_col].nunique(),
    }


def theil_origin(data, access_col, origin_cols=("NAT", "EU_OTH", "OTH")):
    parts = []
    for g in origin_cols:
        sub = data[[access_col, g]].dropna().copy()
        sub = sub[sub[g] > 0].copy()
        if len(sub) == 0:
            continue
        sub["group"] = g
        sub["weight"] = sub[g]
        parts.append(sub[[access_col, "group", "weight"]])
    if not parts:
        return None
    long = pd.concat(parts, ignore_index=True)
    return theil_decompose(long, access_col, "group", "weight")


In [7]:
# ── THEIL ON ACCESSIBILITY LEVELS ─────────────────────────────────
results = []

for scen in SCENARIOS:
    ac = f"A_agg_{scen}"

    overall = weighted_theil_total(df, ac, "T")
    results.append({
        "scenario": scen, "decomposition": "overall",
        "T_total": overall["T_total"],
        "T_between": np.nan, "T_within": np.nan, "T_within_direct": np.nan,
        "between_share_pct": np.nan, "within_share_pct": np.nan,
        "n_cells": overall["n_cells"], "n_pop": overall["n_pop"],
        "n_groups": np.nan,
    })

    r = theil_decompose(df, ac, "municipality", "T")
    results.append({"scenario": scen, "decomposition": "municipality", **r})

    r = theil_decompose(df, ac, "ses_tercile", "T")
    results.append({"scenario": scen, "decomposition": "ses_tercile", **r})

    r = theil_origin(df, ac)
    results.append({"scenario": scen, "decomposition": "origin", **r})

res = pd.DataFrame(results)
mask = res["decomposition"] != "overall"
res.loc[mask, "decomp_check_absdiff"] = np.abs(
    res.loc[mask, "T_within"] - res.loc[mask, "T_within_direct"]
)

res.to_csv("theil_results.csv", index=False)

print(res[[
    "scenario", "decomposition", "T_total", "T_between",
    "T_within", "between_share_pct", "within_share_pct",
]].to_string(index=False))


scenario decomposition  T_total  T_between  T_within  between_share_pct  within_share_pct
       A       overall 0.440065        NaN       NaN                NaN               NaN
       A  municipality 0.440065   0.346900  0.093165          78.829289         21.170711
       A   ses_tercile 0.440065   0.102727  0.337338          23.343645         76.656355
       A        origin 0.439550   0.029526  0.410024           6.717338         93.282662
       D       overall 0.445768        NaN       NaN                NaN               NaN
       D  municipality 0.445768   0.352609  0.093159          79.101490         20.898510
       D   ses_tercile 0.445768   0.103830  0.341938          23.292426         76.707574
       D        origin 0.445251   0.029988  0.415263           6.735174         93.264826


In [8]:
# ── THEIL ON THE GAIN VECTOR  ─────────────────
df["gain"] = df["A_agg_D"] - df["A_agg_A"]

pop_total = df["T"].sum()
pop_zero  = df.loc[df["gain"] <= 0, "T"].sum()
print(f"Population with zero gain: {pop_zero:,.0f} / {pop_total:,.0f} "
      f"({pop_zero/pop_total*100:.1f}%)")
print(f"Cells with zero gain: {(df['gain'] <= 0).sum()} / {len(df)}")

df_pos = df[df["gain"] > 0].copy()

gain_results = []

overall = weighted_theil_total(df_pos, "gain", "T")
gain_results.append({
    "decomposition": "overall",
    "T_total": overall["T_total"],
    "T_between": np.nan, "T_within": np.nan, "T_within_direct": np.nan,
    "between_share_pct": np.nan, "within_share_pct": np.nan,
    "n_cells": overall["n_cells"], "n_pop": overall["n_pop"],
    "n_groups": np.nan,
})

r = theil_decompose(df_pos, "gain", "municipality", "T")
gain_results.append({"decomposition": "municipality", **r})

r = theil_decompose(df_pos, "gain", "ses_tercile", "T")
gain_results.append({"decomposition": "ses_tercile", **r})

r = theil_origin(df_pos, "gain")
gain_results.append({"decomposition": "origin", **r})

gain_res = pd.DataFrame(gain_results)
mask = gain_res["decomposition"] != "overall"
gain_res.loc[mask, "decomp_check_absdiff"] = np.abs(
    gain_res.loc[mask, "T_within"] - gain_res.loc[mask, "T_within_direct"]
)

gain_res.to_csv("theil_gain.csv", index=False)

print("\n" + gain_res[[
    "decomposition", "T_total", "T_between",
    "T_within", "between_share_pct", "within_share_pct",
    "n_cells", "n_groups",
]].to_string(index=False))


Population with zero gain: 43,922 / 646,170 (6.8%)
Cells with zero gain: 1189 / 2795

decomposition  T_total  T_between  T_within  between_share_pct  within_share_pct  n_cells  n_groups
      overall 1.043894        NaN       NaN                NaN               NaN     1170       NaN
 municipality 1.043894   0.629520  0.414375          60.304926         39.695074     1170     100.0
  ses_tercile 1.043894   0.145175  0.898720          13.907031         86.092969     1170       3.0
       origin 1.043582   0.046388  0.997193           4.445088         95.554912     3154       3.0


In [9]:
# ── WHERE ARE THE ZERO-GAIN CELLS? ──────────────────────────────────────
df_zero = df[df["gain"] <= 0].copy()

# by municipality
zero_by_muni  = df_zero.groupby("municipality")["T"].sum().sort_values(ascending=False)
total_by_muni = df.groupby("municipality")["T"].sum()
share_zero    = (zero_by_muni / total_by_muni).sort_values(ascending=False)

# by SES
zero_by_ses  = df_zero.groupby("ses_tercile")["T"].sum()
total_by_ses = df.groupby("ses_tercile")["T"].sum()
share_zero_ses = zero_by_ses / total_by_ses

# by origin
zero_origin  = df_zero[["NAT", "EU_OTH", "OTH"]].sum()
total_origin = df[["NAT", "EU_OTH", "OTH"]].sum()
share_zero_origin = zero_origin / total_origin

print(f"Share with zero gain by origin:\n{share_zero_origin}")


Share with zero gain by origin:
NAT       0.074440
EU_OTH    0.063628
OTH       0.054585
dtype: float64


In [10]:
# ── DIRECTIONAL SUMMARY OF GAINS ────────────────────────────────────────

def summarise(d, group_col, w):
    d = d[d[w] > 0]
    out = d.groupby(group_col).apply(lambda g: pd.Series({
        "pop":        g[w].sum(),
        "baseline_A": np.average(g["A_agg_A"], weights=g[w]),
        "full_D":     np.average(g["A_agg_D"], weights=g[w]),
        "abs_gain":   np.average(g["gain"],    weights=g[w]),
    }))
    out["rel_gain_pct"] = out["abs_gain"] / out["baseline_A"] * 100
    return out

def by_origin(d):
    # Nationality groups are columns, so stack them: one row per cell and group
    parts = [d.assign(origin=g, w=d[g]) for g in ["NAT", "EU_OTH", "OTH"]]
    return summarise(pd.concat(parts), "origin", "w")

def add_zero_share(summary, d, group_col):
    zero = d[(d["gain"] <= 0) & (d["T"] > 0)].groupby(group_col)["T"].sum()
    summary["zero_gain_share_pct"] = (zero / summary["pop"] * 100).reindex(summary.index).fillna(0)
    return summary

# 1) Among beneficiaries (gain > 0) -> Table gain_direction + top gainers
ses_pos    = summarise(df_pos, "ses_tercile", "T")
origin_pos = by_origin(df_pos)
muni_pos   = summarise(df_pos, "municipality", "T").sort_values("abs_gain", ascending=False)

# 2) All populated residents (incl. zero gain) -> "who does not benefit"
muni_all = add_zero_share(summarise(df, "municipality", "T"), df, "municipality")
ses_all  = add_zero_share(summarise(df, "ses_tercile", "T"), df, "ses_tercile")

pd.set_option("display.float_format", "{:.3f}".format)
print("SES, among beneficiaries:\n", ses_pos, "\n")
print("Nationality, among beneficiaries:\n", origin_pos, "\n")
print("Top 10 municipalities by absolute gain (beneficiaries):\n", muni_pos.head(10), "\n")
print("Municipalities with highest zero-gain share (all residents):\n",
      muni_all.sort_values("zero_gain_share_pct", ascending=False).head(15), "\n")
print("Zero-gain share by SES (all residents):\n", ses_all["zero_gain_share_pct"])

ses_pos.to_csv("gain_direction_ses.csv")
origin_pos.to_csv("gain_direction_origin.csv")
muni_pos.to_csv("gain_by_municipality_beneficiaries.csv")
muni_all.to_csv("gain_by_municipality_all_residents.csv")

SES, among beneficiaries:
                    pop  baseline_A  full_D  abs_gain  rel_gain_pct
ses_tercile                                                       
high        199713.000       3.583   3.679     0.095         2.661
low         297377.000       8.722   8.985     0.263         3.011
mid         105158.000       3.096   3.158     0.062         2.014 

Nationality, among beneficiaries:
               pop  baseline_A  full_D  abs_gain  rel_gain_pct
origin                                                       
EU_OTH 204524.221       7.340   7.563     0.223         3.034
NAT    302849.453       4.635   4.756     0.121         2.600
OTH     94563.326       7.716   7.945     0.229         2.965 

Top 10 municipalities by absolute gain (beneficiaries):
                         pop  baseline_A  full_D  abs_gain  rel_gain_pct
municipality                                                           
Luxembourg       125299.000      14.706  15.230     0.524         3.563
Schifflange     

In [11]:
print(muni_all.loc[["Winseler", "Clervaux", "Weiswampach", "Wincrange"]])

                  pop  baseline_A  full_D  abs_gain  rel_gain_pct  \
municipality                                                        
Winseler     1456.000       0.254   0.254     0.000         0.000   
Clervaux     5866.000       0.503   0.503     0.000         0.000   
Weiswampach  2272.000       0.472   0.472     0.000         0.000   
Wincrange    4812.000       0.200   0.200     0.000         0.000   

              zero_gain_share_pct  
municipality                       
Winseler                   12.569  
Clervaux                    8.149  
Weiswampach                22.623  
Wincrange                  32.502  


In [12]:


SENSITIVITY_CASES = {
    "<30 = 1":             1,
    "<30 = 15 (main)":     15,
    "<30 = 29":            29,
    "suppressed excluded": None,   # residents of <30 cells removed entirely
}

def build_df(midpoint):
    o = origins[["GRD_ID", "Pop_grids", "is_suppressed"]].copy()
    if midpoint is None:
        o["T"] = o["Pop_grids"].apply(parse_pop_value)
        o.loc[o["is_suppressed"], "T"] = 0
    else:
        o["T"] = o["Pop_grids"].apply(lambda x: parse_pop_value(x, midpoint=midpoint))

    # Nationality: same imputation rule as the main analysis
    p = o.merge(census[["SPATIAL", "NAT", "EU_OTH", "OTH"]],
                left_on="GRD_ID", right_on="SPATIAL", how="left")
    p["T_census"] = p[["NAT", "EU_OTH", "OTH"]].sum(axis=1)
    need = p["NAT"].isna() | (p["T_census"] == 0)
    for col, share in [("NAT", nat_share), ("EU_OTH", eu_oth_share), ("OTH", oth_share)]:
        p.loc[need, col] = p.loc[need, "T"] * share
    if midpoint is None:
        # remove suppressed residents from the nationality weights as well
        p.loc[p["is_suppressed"], ["NAT", "EU_OTH", "OTH"]] = 0

    d = acc.merge(p[["GRD_ID", "T", "NAT", "EU_OTH", "OTH"]], on="GRD_ID", how="inner")
    d = d.merge(muni, on="GRD_ID", how="inner")
    d["gain"] = d["A_agg_D"] - d["A_agg_A"]
    return d

def run_theil(d):
    rows = []
    analyses = [
        ("level_A", d,                "A_agg_A"),
        ("level_D", d,                "A_agg_D"),
        ("gain",    d[d["gain"] > 0], "gain"),
    ]
    for label, data, col in analyses:
        o = weighted_theil_total(data, col, "T")
        rows.append({"analysis": label, "decomposition": "overall",
                     "T_total": o["T_total"], "between_share_pct": np.nan})
        for grp in ["municipality", "ses_tercile"]:
            r = theil_decompose(data, col, grp, "T")
            rows.append({"analysis": label, "decomposition": grp,
                         "T_total": r["T_total"], "between_share_pct": r["between_share_pct"]})
        r = theil_origin(data, col)
        rows.append({"analysis": label, "decomposition": "origin",
                     "T_total": r["T_total"], "between_share_pct": r["between_share_pct"]})
    return rows

sens_rows, sens_summary = [], []
for case, midpoint in SENSITIVITY_CASES.items():
    d = build_df(midpoint)
    for row in run_theil(d):
        sens_rows.append({"case": case, **row})
    populated = d[d["T"] > 0]
    sens_summary.append({
        "case": case,
        "populated_cells": len(populated),
        "total_pop": populated["T"].sum(),
        "zero_gain_pop_share_pct": populated.loc[populated["gain"] <= 0, "T"].sum()
                                   / populated["T"].sum() * 100,
    })

sens = pd.DataFrame(sens_rows)
case_order = list(SENSITIVITY_CASES)

T_table = sens.pivot_table(index=["analysis", "decomposition"], columns="case",
                           values="T_total")[case_order]
B_table = sens[sens["decomposition"] != "overall"].pivot_table(
    index=["analysis", "decomposition"], columns="case",
    values="between_share_pct")[case_order]
S_table = pd.DataFrame(sens_summary).set_index("case")

pd.set_option("display.float_format", "{:.3f}".format)
print("Population and zero-gain share:\n", S_table, "\n")
print("Theil index (T_total):\n", T_table, "\n")
print("Between-group share (%):\n", B_table)

sens.to_csv("theil_sensitivity_suppressed.csv", index=False)
S_table.to_csv("theil_sensitivity_suppressed_summary.csv")

Population and zero-gain share:
                      populated_cells  total_pop  zero_gain_pop_share_pct
case                                                                    
<30 = 1                         1636 638232.000                    6.279
<30 = 15 (main)                 1636 646170.000                    6.797
<30 = 29                        1636 654108.000                    7.303
suppressed excluded             1069 637665.000                    6.241 

Theil index (T_total):
 case                    <30 = 1  <30 = 15 (main)  <30 = 29  \
analysis decomposition                                       
gain     municipality     1.038            1.044     1.049   
         origin           1.041            1.044     1.046   
         overall          1.038            1.044     1.049   
         ses_tercile      1.038            1.044     1.049   
level_A  municipality     0.431            0.440     0.449   
         origin           0.434            0.440     0.445   
       

In [13]:
partial = census["NAT"].notna() & census[["EU_OTH", "OTH"]].isna().any(axis=1)
print("Cells with NAT present but EU_OTH or OTH missing:", partial.sum())

Cells with NAT present but EU_OTH or OTH missing: 307


In [14]:
# ── CHECK: CELLS WITH PARTLY MISSING NATIONALITY COUNTS ────────────────

groups = ["NAT", "EU_OTH", "OTH"]
partial_df = df["NAT"].notna() & df[["EU_OTH", "OTH"]].isna().any(axis=1)

# 1) Diagnosis
pp = df.loc[partial_df & (df["T"] > 0)]
known = pp[groups].sum(axis=1)
populated = df[df["T"] > 0]
print("Populated cells with partly missing counts :", len(pp))
print("  Missing EU_OTH:", pp["EU_OTH"].isna().sum(), "| Missing OTH:", pp["OTH"].isna().sum())
print(f"  Residents in these cells (T)             : {pp['T'].sum():,.0f}")
print(f"  Residents counted in census groups       : {known.sum():,.0f}")
print(f"  Residents not assigned to any group      : {(pp['T'] - known).clip(lower=0).sum():,.0f}")
print(f"  Mean baseline accessibility, these cells : {np.average(pp['A_agg_A'], weights=pp['T']):.3f}")
print(f"  Mean baseline accessibility, all cells   : {np.average(populated['A_agg_A'], weights=populated['T']):.3f}\n")

# 2) Sensitivity: give the unassigned residents (T minus census groups)
#    to the missing group(s); if both are missing, split by national shares.
df_fill = df.copy()
r = df_fill.loc[partial_df]
resid = (r["T"] - r[groups].sum(axis=1)).clip(lower=0)
both = r["EU_OTH"].isna() & r["OTH"].isna()
eu_part = eu_oth_share / (eu_oth_share + oth_share)
fill_eu  = pd.Series(np.where(both, resid * eu_part,       resid), index=r.index)
fill_oth = pd.Series(np.where(both, resid * (1 - eu_part), resid), index=r.index)
df_fill.loc[partial_df, "EU_OTH"] = r["EU_OTH"].fillna(fill_eu)
df_fill.loc[partial_df, "OTH"]    = r["OTH"].fillna(fill_oth)

comp = []
for label, col in [("level_A", "A_agg_A"), ("level_D", "A_agg_D"), ("gain", "gain")]:
    d_main = df[df["gain"] > 0] if label == "gain" else df
    d_fill = df_fill[df_fill["gain"] > 0] if label == "gain" else df_fill
    m, f = theil_origin(d_main, col), theil_origin(d_fill, col)
    comp.append({"analysis": label,
                 "T_main": m["T_total"], "T_filled": f["T_total"],
                 "between_pct_main": m["between_share_pct"],
                 "between_pct_filled": f["between_share_pct"]})

pd.set_option("display.float_format", "{:.3f}".format)
print("Nationality decomposition: main vs filled\n", pd.DataFrame(comp).set_index("analysis"))

Populated cells with partly missing counts : 303
  Missing EU_OTH: 140 | Missing OTH: 194
  Residents in these cells (T)             : 12,650
  Residents counted in census groups       : 11,808
  Residents not assigned to any group      : 1,112
  Mean baseline accessibility, these cells : 0.644
  Mean baseline accessibility, all cells   : 5.626

Nationality decomposition: main vs filled
           T_main  T_filled  between_pct_main  between_pct_filled
analysis                                                        
level_A    0.440     0.441             6.717               6.607
level_D    0.445     0.447             6.735               6.625
gain       1.044     1.044             4.445               4.409


In [15]:
# the gain inequality for the full population (including zero-gain locations)
df_all = df[df["gain"] >= 0]   # all residents, including zero gain
print("Overall:", round(weighted_theil_total(df_all, "gain", "T")["T_total"], 3))
for grp in ["municipality", "ses_tercile"]:
    r = theil_decompose(df_all, "gain", grp, "T")
    print(grp, round(r["T_total"], 3), round(r["between_share_pct"], 1))
r = theil_origin(df_all, "gain")
print("origin", round(r["T_total"], 3), round(r["between_share_pct"], 1))

Overall: 1.114
municipality 1.114 58.8
ses_tercile 1.114 13.3
origin 1.114 4.4


In [16]:
print(ses_all)


                   pop  baseline_A  full_D  abs_gain  rel_gain_pct  \
ses_tercile                                                          
high        214981.000       3.330   3.418     0.089         2.660   
low         317270.000       8.175   8.421     0.246         3.011   
mid         113919.000       2.859   2.917     0.058         2.013   

             zero_gain_share_pct  
ses_tercile                       
high                       7.102  
low                        6.270  
mid                        7.691  


In [17]:
# SES tercile of each municipality (code labels: low = high-SES, high = low-SES)
label = {"low": "High-SES", "mid": "Mid-SES", "high": "Low-SES"}

muni_ses = (df[["municipality", "ses_tercile"]]
            .drop_duplicates()
            .set_index("municipality")["ses_tercile"]
            .map(label))

names = ["Luxembourg", "Bertrange", "Strassen",
         "Schifflange", "Differdange", "Esch-sur-Alzette"]
print(muni_ses.loc[names])

municipality
Luxembourg          High-SES
Bertrange           High-SES
Strassen            High-SES
Schifflange          Low-SES
Differdange          Low-SES
Esch-sur-Alzette     Low-SES
Name: ses_tercile, dtype: str


In [18]:
print(df.groupby("municipality")["ses_tercile"].nunique().max())

1


In [19]:
print(muni_ses.sort_values().to_string())

municipality
Schuttrange             High-SES
Garnich                 High-SES
Sandweiler              High-SES
Mamer                   High-SES
Flaxweiler              High-SES
Strassen                High-SES
Niederanven             High-SES
Feulen                  High-SES
Betzdorf                High-SES
Koerich                 High-SES
Kehlen                  High-SES
Kopstal                 High-SES
Bertrange               High-SES
Walferdange             High-SES
Biwer                   High-SES
Habscht                 High-SES
Lorentzweiler           High-SES
Junglinster             High-SES
Mertzig                 High-SES
Helperknapp             High-SES
Saeul                   High-SES
Fischbach               High-SES
Bech                    High-SES
Useldange               High-SES
Heffingen               High-SES
Steinsel                High-SES
Erpeldange-sur-Sûre     High-SES
Ell                     High-SES
Luxembourg              High-SES
Dippach                 High-S

In [20]:
print(df.loc[(df["T"] > 0) & (df["gain"] <= 0), "A_agg_A"].describe())

# So the residents who gain nothing are residents who have almost no PT accessibility to begin with.

count   466.000
mean      0.009
std       0.036
min       0.000
25%       0.000
50%       0.000
75%       0.000
max       0.320
Name: A_agg_A, dtype: float64


In [21]:
nb = df[(df["T"] > 0) & (df["gain"] <= 0)]
print(np.average(nb["A_agg_A"], weights=nb["T"]))
print((nb.loc[nb["A_agg_A"] == 0, "T"].sum() / nb["T"].sum()) * 100)

# Non-beneficiaries have a mean baseline of 0.007, and 94.1% of them have exactly zero accessibility.

0.007283222800124066
94.10545967852102


In [23]:
origin_all = by_origin(df)
zero = df.loc[df["gain"] <= 0, ["NAT", "EU_OTH", "OTH"]].sum()
total = df[["NAT", "EU_OTH", "OTH"]].sum()
origin_all["zero_gain_share_pct"] = zero / total * 100
print(origin_all)

              pop  baseline_A  full_D  abs_gain  rel_gain_pct  \
origin                                                          
EU_OTH 218422.034       6.873   7.082     0.209         3.034   
NAT    327206.872       4.291   4.402     0.112         2.600   
OTH    100023.094       7.295   7.511     0.216         2.965   

        zero_gain_share_pct  
origin                       
EU_OTH                6.363  
NAT                   7.444  
OTH                   5.459  
